<a href="https://colab.research.google.com/github/Minenhlekhuzwayo/Satellite-Data-Assimilation-For-Hybrid-Crop-Yield-Prediction/blob/main/HonoursProject.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### 1. NASA WEATHER DATA PREPROCESSING

In [ ]:
!pip install refet
import pandas as pd
import refet

# Load data
df_weather = pd.read_csv('weatherDataNASA.csv')

# Remove hidden spaces
df_weather.columns = df_weather.columns.str.strip()

# Rename columns
df_weather = df_weather.rename(columns={
    'DOY':'dateOfYear',
    'ALLSKY_SFC_SW_DWN': 'solarRadiation',
    'T2M_MAX' : 'MaxTemp',
    'T2M_MIN' : 'MinTemp',
    'PRECTOTCORR': 'Precipitation',
    'RH2M': 'RelativeHumidity',
    'WS2M': 'WindSpeed',
    'GWETROOT': 'RootMoisture'
})

# Create Date column
df_weather['Date'] = (
    pd.to_datetime(df_weather['YEAR'], format='%Y') +
    pd.to_timedelta(df_weather['dateOfYear'] - 1, unit='D')
)

#df_weather['Date'] = df_weather['Date'].dt.strftime('%Y/%m/%d')
df_weather['Day'] = df_weather['Date'].dt.day
df_weather['Month'] = df_weather['Date'].dt.month
df_weather['Year'] = df_weather['Date'].dt.year


# Constants
altitude = 350
latitude = -33.5

eto_list = []

# ETo calculation
for index, row in df_weather.iterrows():

    eto = refet.Daily(
        tmin=row['MinTemp'],
        tmax=row['MaxTemp'],
        rs=row['solarRadiation'],
        uz=row['WindSpeed'],
        zw = 2,
        elev=altitude,
        lat=latitude,
        doy=row['dateOfYear'],
        tdew = row['MinTemp']
    ).eto()

    eto_list.append(float(eto)) # Extract the scalar value

# Add ETo column, renaming it to 'ReferenceET' as expected by AquaCrop's prepare_weather
df_weather['ReferenceET'] = eto_list

# 1. Full dataset (for documentation/analysis) with ETo
df_final_full_weather = df_weather[
    ['Date', 'MinTemp', 'MaxTemp', 'Precipitation', 'solarRadiation', 'RelativeHumidity', 'WindSpeed', 'RootMoisture','ReferenceET']
]

# Export full dataset with ETo
df_weather.to_csv('full_weather_with_eto.txt', sep=' ', index=False)



# 2. Final AquaCrop dataframe
df_final_weather_aqua_inputs = df_weather[
    ['Day', 'Month', 'Year','MinTemp', 'MaxTemp', 'Precipitation', 'ReferenceET'] # Use 'ReferenceET' here
]

# Export
df_final_weather_aqua_inputs.to_csv(
    'aquaCropWeatherData.txt',
    sep=' ',
    index=False
)

# Preview
print(df_final_weather_aqua_inputs.head())

### 2. RUNNING AQUACROP MODEL

In [ ]:
!pip install aquacrop
from aquacrop import AquaCropModel, Soil, Crop, InitialWaterContent
from aquacrop.utils import prepare_weather

weather_df = prepare_weather('aquaCropWeatherData.txt')

grape_crop = Crop('Maize', planting_date='10/01')
grape_crop.Maturity=240
grape_crop.Zmax=2.0
grape_crop.CGC=0.004
grape_crop.CDG=0.002
grape_crop.HI0=0.35

model_os = AquaCropModel(
    sim_start_time='2010/01/01',
    sim_end_time='2024/12/31',
    weather_df=weather_df,
    soil=Soil('SandyLoam'),
    crop=grape_crop,
    initial_water_content=InitialWaterContent(value=['FC'])
)

model_os.run_model(till_termination=True)

results = model_os.get_simulation_results() # Returns seasonal summary output

print(results)

# Crop growth outputs
print("--------------------Crop Growth Output--------------------")
crop_growth = model_os._outputs.crop_growth
print(crop_growth)

# Water flux outputs
print("--------------------Water Flux Output---------------------")
water_flux = model_os._outputs.water_flux
print(water_flux)

# Soil water outputs
print("--------------------Water Storage Outputs--------------------")
water_storage = model_os._outputs.water_storage
print(water_storage)

# EXPORTING OUTPUTS
results.to_csv('results_Maize.csv')
crop_growth.to_csv('crop_growth_Maize.csv')
water_flux.to_csv('water_flux_Maize.csv')
water_storage.to_csv('water_storage_Maize.csv')

In [ ]:
print(dir(model_os._outputs))

In [ ]:
crop_growth = model_os._outputs.crop_growth

print(crop_growth.head)

### 3. SENTINEL-2 FEATURE EXTRACTION FOR HYBRID YIELD PREDICTION

In [ ]:
# FEATURES
# - NDVI
# - EVI
# - NDWI
# - NDRE
# - Raw bands (B4, B5, B8, B11)
# PERIOD:
         # September -> March
# OUTPUT:
         # CSV-ready feature table for ML / data assimilation



# 1. INSTALL & IMPORT LIBRARIES
!pip install earthengine-api geemap
import ee
import pandas as pd


# 2. AUTHENTICATE & INITIALIZE EARTH ENGINE
ee.Authenticate()
ee.Initialize(project='vineyard-yield-project-400201')
print("Earth Engine initialized successfully!")


# 3. STUDY AREA
study_area = ee.Geometry.Rectangle([
    19.30,   # min longitude
    -33.60,  # min latitude
    19.90,   # max longitude
    -33.30   # max latitude
])



# 4. LOADING SENTINEL-2 LEVEL-2A DATA
s2 = (
    ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
    .filterBounds(study_area)
    .filterDate('2015-09-01', '2024-03-31')
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20))
)



# 5. CLOUD MASKING FUNCTION
def mask_clouds(image):

    qa = image.select('QA60')

    cloud_bit_mask = 1 << 10
    cirrus_bit_mask = 1 << 11

    mask = (
        qa.bitwiseAnd(cloud_bit_mask).eq(0)
        .And(qa.bitwiseAnd(cirrus_bit_mask).eq(0))
    )

    masked =  image.updateMask(mask).divide(10000)
    return masked.copyProperties(
        image,
        image.propertyNames()
    )


# Apply cloud mask FIRST
s2 = s2.map(mask_clouds)




# 6. VEGETATION INDICES CALCULATIONS
def add_indices(image):

    ndvi = image.normalizedDifference(['B8', 'B4']) \
        .rename('NDVI')

    ndwi = image.normalizedDifference(['B8', 'B11']) \
        .rename('NDWI')

    ndre = image.normalizedDifference(['B8', 'B5']) \
        .rename('NDRE')

    evi = image.expression(
        '2.5 * ((NIR - RED) / (NIR + 6 * RED - 7.5 * BLUE + 1))',
        {
            'NIR': image.select('B8'),
            'RED': image.select('B4'),
            'BLUE': image.select('B2')
        }
    ).rename('EVI')

    # IMPORTANT:
    # add indices back to image
    return image.addBands([ndvi, ndwi, ndre, evi]) \
      .copyProperties(
          image,
          image.propertyNames()
      )


# Apply index calculation
s2_with_indices = s2.map(add_indices)




# 7. SELECTING REQUIRED FEATURES
bands = [
    'NDVI',
    'NDWI',
    'NDRE',
    'EVI',
    'B4',
    'B5',
    'B8',
    'B11'
]




# 8. EXTRACTING MEAN VALUES OVER STUDY AREA
def extract_features(image):

    stats = image.select(bands).reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=study_area,
        scale=10,
        maxPixels=1e13
    )

    feature = ee.Feature(
        None,
        stats
    ).set(
        'date',
        image.date().format('YYYY-MM-dd')
    )

    return feature


# Convert images to features
features = s2_with_indices.map(extract_features)

feature_collection = ee.FeatureCollection(features)




# 9. EXPORT TO CSV
task = ee.batch.Export.table.toDrive(
    collection=feature_collection,
    description='Sentinel2_Vineyard_Features',
    folder='satelliteDataFolder',
    fileFormat='CSV'
)

task.start()

print(task.status())

In [ ]:
print(task.status())

### 4. MERGING THE DATASETS: HYBRID DATASET CREATION FOR ML

In [ ]:
# =========================================================
# HYBRID DATASET CREATION FOR ML
# Satellite Data Assimilation for Hybrid Yield Prediction
# =========================================================

import pandas as pd

# =========================================================
# 1. LOAD DATASETS
# =========================================================

crop_growth = pd.read_csv("crop_growth_Maize.csv")

water_storage = pd.read_csv("water_storage_Maize.csv")

water_flux = pd.read_csv("water_flux_Maize.csv")

satellite_data = pd.read_csv(
    "Sentinel2_Vineyard_Features.csv"
)

weather_data = pd.read_csv(
    "aquaCropWeatherData.txt",
    sep=r"\s+"
)

# =========================================================
# 2. CREATE DATE COLUMN FOR AQUACROP OUTPUTS
# =========================================================

# AquaCrop simulation start date
start_date = pd.to_datetime("2010-01-01")

# Create dates from time_step_counter
crop_growth['Date'] = (
    start_date +
    pd.to_timedelta(
        crop_growth['time_step_counter'],
        unit='D'
    )
)

water_storage['Date'] = (
    start_date +
    pd.to_timedelta(
        water_storage['time_step_counter'],
        unit='D'
    )
)

water_flux['Date'] = (
    start_date +
    pd.to_timedelta(
        water_flux['time_step_counter'],
        unit='D'
    )
)

# =========================================================
# 3. PROCESS SATELLITE DATES
# =========================================================

satellite_data['date'] = pd.to_datetime(
    satellite_data['date']
)

# Rename satellite date column
satellite_data.rename(
    columns={'date': 'Date'},
    inplace=True
)

# =========================================================
# 4. PROCESS WEATHER DATES
# =========================================================

weather_data['Date'] = pd.to_datetime(
    weather_data[['Year', 'Month', 'Day']]
)

# =========================================================
# 5. REMOVE UNNECESSARY COLUMNS
# =========================================================

# Remove index columns
crop_growth.drop(
    columns=['Unnamed: 0'],
    inplace=True,
    errors='ignore'
)

water_storage.drop(
    columns=['Unnamed: 0'],
    inplace=True,
    errors='ignore'
)

water_flux.drop(
    columns=['Unnamed: 0'],
    inplace=True,
    errors='ignore'
)

# Remove unnecessary satellite columns
satellite_data.drop(
    columns=['system:index', '.geo'],
    inplace=True,
    errors='ignore'
)

# =========================================================
# 6. SELECT IMPORTANT FEATURES
# =========================================================

# -------------------------
# Crop Growth Features
# -------------------------

crop_growth = crop_growth[
    [
        'Date',
        'dap',
        'gdd_cum',
        'z_root',
        'canopy_cover',
        'biomass',
        'harvest_index',
        'DryYield',
        'YieldPot'
    ]
]

# -------------------------
# Water Storage Features
# -------------------------

water_storage = water_storage[
    [
        'Date',
        'th1',
        'th2',
        'th3',
        'th4'
    ]
]

# -------------------------
# Water Flux Features
# -------------------------

water_flux = water_flux[
    [
        'Date',
        'Wr',
        'Infl',
        'Runoff',
        'DeepPerc',
        'Es',
        'Tr',
        'TrPot'
    ]
]

# -------------------------
# Satellite Features
# -------------------------

satellite = satellite_data[
    [
        'Date',
        'NDVI',
        'EVI',
        'NDRE',
        'NDWI',
        'B4',
        'B5',
        'B8',
        'B11'
    ]
]

# -------------------------
# Weather Features
# -------------------------

weather = weather_data[
    [
        'Date',
        'MaxTemp',
        'MinTemp',
        'Precipitation',
        'ReferenceET'
    ]
]

# =========================================================
# 7. REMOVE DUPLICATE DATES
# =========================================================

crop_growth = crop_growth.drop_duplicates(
    subset='Date'
)

water_storage = water_storage.drop_duplicates(
    subset='Date'
)

water_flux = water_flux.drop_duplicates(
    subset='Date'
)

weather = weather.drop_duplicates(
    subset='Date'
)

# Average duplicate satellite dates
satellite = satellite.groupby(
    'Date'
).mean().reset_index()

# =========================================================
# 8. MERGE AQUACROP DATASETS
# =========================================================

aquacrop = pd.merge(
    crop_growth,
    water_storage,
    on='Date',
    how='inner'
)

aquacrop = pd.merge(
    aquacrop,
    water_flux,
    on='Date',
    how='inner'
)

# =========================================================
# 9. MERGE WEATHER DATA
# =========================================================

aquacrop_weather = pd.merge(
    aquacrop,
    weather,
    on='Date',
    how='inner'
)

# =========================================================
# 10. FILTER TO SATELLITE PERIOD
# =========================================================

aquacrop_weather = aquacrop_weather[
    aquacrop_weather['Date'] >= '2015-12-18'
]

# =========================================================
# 11. SORT DATASETS
# =========================================================

aquacrop_weather = aquacrop_weather.sort_values(
    'Date'
)

satellite = satellite.sort_values(
    'Date'
)

# =========================================================
# 12. MERGE SATELLITE DATA
# USING NEAREST DATE MATCHING
# =========================================================

final_dataset = pd.merge_asof(
    aquacrop_weather,
    satellite,
    on='Date',
    direction='nearest'
)

# =========================================================
# 13. HANDLE MISSING VALUES
# =========================================================

# Interpolate missing values
final_dataset = final_dataset.interpolate()

# Remove remaining missing values
final_dataset = final_dataset.dropna()

# =========================================================
# 14. CHECK FINAL DATASET
# =========================================================

print("FINAL DATASET SHAPE:")
print(final_dataset.shape)

print("\nFINAL DATASET COLUMNS:")
print(final_dataset.columns)

print("\nFIRST 5 ROWS:")
print(final_dataset.head())

# =========================================================
# 15. SAVE FINAL HYBRID DATASET
# =========================================================

final_dataset.to_csv(
    "Hybrid_ML_Dataset.csv",
    index=False
)

print("\nHybrid dataset saved successfully!")

# =========================================================
# 16. PREPARE DATA FOR MACHINE LEARNING
# =========================================================

# Target variable
y = final_dataset['DryYield']

# Feature variables
X = final_dataset.drop(
    columns=['DryYield', 'Date']
)

print("\nFEATURE MATRIX SHAPE:")
print(X.shape)

print("\nTARGET VECTOR SHAPE:")
print(y.shape)

# =========================================================
# 17. TRAIN TEST SPLIT
# =========================================================

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("\nTRAINING SET SHAPE:")
print(X_train.shape)

print("\nTEST SET SHAPE:")
print(X_test.shape)

# =========================================================
# 18. RANDOM FOREST MODEL
# =========================================================

from sklearn.ensemble import RandomForestRegressor

from sklearn.metrics import (
    mean_absolute_error,
    r2_score
)

# Create model
model = RandomForestRegressor(
    n_estimators=100,
    random_state=42
)

# Train model
model.fit(X_train, y_train)

# Predictions
predictions = model.predict(X_test)

# =========================================================
# 19. MODEL EVALUATION
# =========================================================

mae = mean_absolute_error(
    y_test,
    predictions
)

r2 = r2_score(
    y_test,
    predictions
)

print("\nMODEL PERFORMANCE")
print("------------------")

print(f"MAE: {mae:.4f}")

print(f"R² Score: {r2:.4f}")

# =========================================================
# 20. FEATURE IMPORTANCE
# =========================================================

feature_importance = pd.DataFrame({
    'Feature': X.columns,
    'Importance': model.feature_importances_
})

feature_importance = feature_importance.sort_values(
    by='Importance',
    ascending=False
)

print("\nTOP IMPORTANT FEATURES")
print("----------------------")

print(feature_importance)

### 4.2 VISUALISING OUTPUTS

### 4.2.1 NDVI TIME SERIES

In [ ]:
# NDVI TIME SERIES

import matplotlib.pyplot as plt

plt.figure(figsize=(12,6))

plt.plot(
    final_dataset['Date'],
    final_dataset['NDVI']
)

plt.xlabel("Date")
plt.ylabel("NDVI")
plt.title("NDVI Time Series")

plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

### 4.2.2 CANOPY COVER VS NDVI

In [ ]:
# CANOPY COVER VS NDVI

plt.figure(figsize=(8,6))

plt.scatter(
    final_dataset['NDVI'],
    final_dataset['canopy_cover']
)

plt.xlabel("NDVI")
plt.ylabel("Canopy Cover")

plt.title("NDVI vs AquaCrop Canopy Cover")

plt.show()

### 4.2.3 BIOMASS OVER TIME

In [ ]:
plt.figure(figsize=(12,6))

plt.plot(
    final_dataset['Date'],
    final_dataset['biomass']
)

plt.xlabel("Date")
plt.ylabel("Biomass")

plt.title("Biomass Growth Over Time")

plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

### 4.2.4 SOIL WATER CONTENT

In [ ]:
plt.figure(figsize=(12,6))

plt.plot(
    final_dataset['Date'],
    final_dataset['th1'],
    label='th1'
)

plt.plot(
    final_dataset['Date'],
    final_dataset['th2'],
    label='th2'
)

plt.xlabel("Date")
plt.ylabel("Soil Water Content")

plt.title("Soil Water Dynamics")

plt.legend()

plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

### 4.2.5 FEATURE IMPORTANCE GRAPH

In [ ]:
plt.figure(figsize=(10,6))

plt.barh(
    feature_importance['Feature'][:10],
    feature_importance['Importance'][:10]
)

plt.xlabel("Importance")
plt.ylabel("Feature")

plt.title("Top 10 Important Features")

plt.gca().invert_yaxis()

plt.show()

### 4.2.6 PREDICTED VS ACTUAL YIELD

In [ ]:
plt.figure(figsize=(8,6))

plt.scatter(
    y_test,
    predictions
)

plt.xlabel("Actual Yield")
plt.ylabel("Predicted Yield")

plt.title("Predicted vs Actual Yield")

plt.plot(
    [y_test.min(), y_test.max()],
    [y_test.min(), y_test.max()]
)

plt.show()

### 4.2.7 CORRELATION HEATMAP

In [ ]:
import seaborn as sns

plt.figure(figsize=(14,10))

sns.heatmap(
    final_dataset.corr(numeric_only=True),
    cmap='coolwarm'
)

plt.title("Feature Correlation Heatmap")

plt.show()